In [1]:
from qiskit import QuantumCircuit
from pauli import *
from experiment import *
from experiment_runner import *
from summarize import *
from visualisation import *
import random
from qiskit import transpile

/var/folders/3s/t7fx2g4s1g5f94plqhz5x1mm0000gn/T/ipykernel_99499/1075277346.py:1: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit import QuantumCircuit


In [2]:
#building the dj oracle
identity = QuantumCircuit(2)
identity.cx(0,1)
identity_gate = identity.to_gate()
identity_gate.name = "identity"


complement = QuantumCircuit(2)
complement.x(0)
complement.cx(0,1)
complement.x(0)
complement_gate = complement.to_gate()
complement_gate.name = "complement"


always_1 = QuantumCircuit(2)
always_1.x(1)
always_1_gate = always_1.to_gate()
always_1_gate.name = "always_1"


always_0 = QuantumCircuit(2)
always_0_gate = always_0.to_gate()
always_0_gate.name = "always_0"

oracle = random.choice([identity_gate,complement_gate,always_1_gate,always_0_gate])

In [3]:
#constructing the circuit algorithm
qc = QuantumCircuit(2,1)
qc.h(0)
qc.x(1)
qc.h(1)
qc.barrier()
qc.append(oracle,[0,1])
qc.barrier()
qc.h(0)
qc.measure(0,0)
qc.draw(output='text')

┌───┐      ░ ┌───────────┐ ░ ┌───┐┌─┐
q_0: ┤ H ├──────░─┤0          ├─░─┤ H ├┤M├
     ├───┤┌───┐ ░ │  always_1 │ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤1          ├─░───────╫─
     └───┘└───┘ ░ └───────────┘ ░       ║ 
c: 1/═══════════════════════════════════╩═
                                        0

In [4]:
#testing that the circuit works correctly
test_sim = AerSimulator()
transpiled_qc = transpile(qc,test_sim)
job = test_sim.run(transpiled_qc,shots=1000)
result = job.result()
counts = result.get_counts()
print(counts)
print(oracle.name)

/Users/aditya/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


{'0': 1000}
always_1


In [5]:
no_noise = PauliChannel(x=0, y=0, z=0, i=1)

dj_experiment = Experiment(
    name="Deutsch-Jozsa - Identity Oracle",
    circuit=qc,
    noise_model=no_noise,
    shots=1000
)

runner = ExperimentRunner()
result = runner.run(dj_experiment)

print(summarize_results(result))

{'0': 1000}
{'0': 1000}
1.0


In [15]:
noise = PauliChannel(x=0.01,y=0.0,z=0.01,i=0.98)
noisy_dj = Experiment(
    name="Deutsch-Jozsa - Identity - Pauli Noise",
    circuit=qc,
    noise_model=noise,
    shots=1000
)

result2 = runner.run(noisy_dj)
print(summarize_results(result2))


===== Experiment Summary =====
Experiment Name = Deutsch-Jozsa - Identity - Pauli Noise
Circuit Depth = 5
Gate Counts = 6
Shots = 1000
Success Probability = 0.957
Countshift:
1:4.30%
0:-4.30%
